> * Name: <span style="color:red"> Rwan Ashraf Abdelazim </span>

> * ID: <span style="color:red"> 221001757</span>

> * Lab: <span style="color:red"> 01b </span>




# **Tasks**

## 1- Data loading:


In [1]:
import pandas as pd
df = pd.read_csv("ar_reviews_100k.tsv", sep='\t')
df.head()


,label,text
0,Positive,ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...
1,Positive,أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...
2,Positive,هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...
3,Positive,خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...
4,Positive,ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...


## 2- Perform data cleaning and tokenization:

- Remove Arabic diacritics (التشكيل)

- Compile a regex pattern that matches: Shadda (ّ), Fatha (َ/ً), Damma (ُ/ٌ), Kasra (ِ/ٍ), Sukun (ْ), and Tatweel (ـ). Use re.sub to delete them from the text.

- Normalize letter variants:

    - Replace [إأآا] → ا
    
    - Replace ى → ي
    
    - Replace ؤ → ء
    
    - Replace ئ → ء
    
    - Replace ة → ه

- Remove punctuation.

- Use a regex like r"[^\w\s]" to remove any character that is not a word character or whitespace.

Return the cleaned text as a new column in the df. Name this column "cleaned_text".

In [10]:
import pyarabic.araby as araby
from pyarabic.araby import strip_tashkeel, strip_tatweel
from pyarabic.araby import tokenize as araby_tokenize
import re

In [14]:
diacritics_pattern = re.compile(
    r'[\u064B-\u0652\u0640]'  # \u064B-\u0652 covers Fatha, Damma, Kasra, Shadda, Sukun
)  # \u0640 is Tatweel (Kashida)

In [11]:
def normalize(sentence):
    # re.sub(pattern (old), replacement (new), string) -> English
    # re.sub(new, old, string) -> Arabic, except for []
    
    # if the pattern is not found, the text is returned unchanged
    sentence = re.sub("[إأآا]", "ا", sentence)
    sentence = re.sub("ى", "ي", sentence)
    sentence = re.sub("ؤ", "ء", sentence)
    sentence = re.sub("ئ", "ء", sentence)
    sentence = re.sub("ة", "ه", sentence)
    return sentence

In [16]:
def clean_arabic_text(text):
    if not isinstance(text, str):
        return ""
    
    text = araby.strip_tashkeel(text)
    text = normalize(text)
    text = re.sub(r"[^\w\s]", "", text)    
    text = diacritics_pattern.sub('', text)
    return text

df['cleaned_text'] = df["text"].apply(clean_arabic_text)

df[['cleaned_text']].head()

,cleaned_text
0,ممتاز نوعا ما النظافه والموقع والتجهيز والشاط...
1,احد اسباب نجاح الامارات ان كل شخص في هذه الدول...
2,هادفه وقويه تنقلك من صخب شوارع القاهره الي هد...
3,خلصنا مبدءيا اللي مستني ابهار زي الفيل الازرق...
4,ياسات جلوريا جزء لا يتجزا من دبي فندق متكامل ...


## 3- Train Naive Bayes to classify the given dataset.

In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer

In [33]:
data = pd.read_csv('data.csv')
X = df['cleaned_text']
y = df['label']

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [35]:
vectorizer = CountVectorizer() #  Converts the raw text into a matrix of word counts.
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

## 4- Use the trained NB model to infer the sentiment of the following sentences:

* أعجبني الجو والطعام والموظفين
* لم يعجبني المكان ولا الجو
* أعجبني الطعام ولم تعجبني طريقة تعامل الموظفين

In [36]:
model = MultinomialNB()
model.fit(X_train_vectorized, y_train)

MultinomialNB()

In [37]:
sentences = [
    "أعجبني الجو والطعام والموظفين",
    "لم يعجبني المكان ولا الجو",
    "أعجبني الطعام ولم تعجبني طريقة تعامل الموظفين"
]
X_new = vectorizer.transform(sentences)
predictions = model.predict(X_new)

In [39]:
for sentence, label in zip(sentences, predictions):
    print(f"Sentence: {sentence}")
    print(f"Predicted Label: {label}")
    print("-" * 40)

Sentence: أعجبني الجو والطعام والموظفين
Predicted Label: Positive
----------------------------------------
Sentence: لم يعجبني المكان ولا الجو
Predicted Label: Negative
----------------------------------------
Sentence: أعجبني الطعام ولم تعجبني طريقة تعامل الموظفين
Predicted Label: Negative
----------------------------------------


In [40]:
y_pred = model.predict(X_test_vectorized)
y_pred

array(['Negative', 'Negative', 'Negative', ..., 'Positive', 'Positive',
       'Negative'], dtype='<U8')